# Flood Susceptibility Model — Data Preparation & Machine Learning
**Study Area:** Nairobi Watershed, Kenya  
**Period:** 2022–2024  
**Repository:** https://github.com/[your-username]/flood-risk-nairobi

This notebook:
1. Loads GeoTIFF layers exported from GEE
2. Aligns all rasters to a common grid
3. Normalises layers 0–1
4. Trains a Random Forest classifier
5. Validates with ROC-AUC and SHAP
6. Exports flood probability rasters

**Run `02_flood_hotspot_mapping.ipynb` after this notebook to generate the final flood risk map.**

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES (run once)
# pip install rasterio geopandas scikit-learn shap matplotlib seaborn pandas numpy scipy
# ============================================================

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio.warp import reproject, Resampling
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.utils import resample
warnings.filterwarnings('ignore')

print('Libraries loaded')

In [ ]:
# ============================================================
# CONFIGURATION — update DATA_DIR to your local folder
# ============================================================
DATA_DIR = r'C:/Users/YourName/GEE_Exports'  # <-- CHANGE THIS

files = {
    'elevation':       'dem_alos_nairobi.tif',
    'slope':           'slope_nairobi.tif',
    'twi':             'twi_nairobi.tif',
    'rainfall':        'rainfall_p95_nairobi.tif',
    'ndvi':            'ndvi_nairobi.tif',
    'dist_river':      'dist_to_river_nairobi.tif',
    'dist_impervious': 'dist_to_impervious_nairobi.tif',
    'clay':            'clay_nairobi.tif',
    'landcover':       'landcover_nairobi.tif',
    'flood_label':     'flood_label_sar_nairobi.tif',
}
TRAINING_CSV = os.path.join(DATA_DIR, 'training_data_nairobi.csv')

print('File check:')
for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    print(f'  {"OK" if os.path.exists(path) else "MISSING":8} {name}')

In [ ]:
# ============================================================
# ALIGN ALL RASTERS TO DEM REFERENCE GRID
# ============================================================
ALIGNED_DIR = os.path.join(DATA_DIR, 'aligned')
os.makedirs(ALIGNED_DIR, exist_ok=True)
ref_path = os.path.join(DATA_DIR, files['elevation'])

def align_raster(src_path, ref_path, out_path):
    with rasterio.open(ref_path) as ref:
        ref_crs, ref_tr, ref_h, ref_w = ref.crs, ref.transform, ref.height, ref.width
    with rasterio.open(src_path) as src:
        if src.crs == ref_crs and src.height == ref_h and src.width == ref_w:
            return src_path
        data = np.zeros((1, ref_h, ref_w), dtype=src.meta['dtype'])
        reproject(source=rasterio.band(src, 1), destination=data,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref_tr, dst_crs=ref_crs,
                  resampling=Resampling.bilinear)
        meta = src.meta.copy()
        meta.update({'crs': ref_crs, 'transform': ref_tr, 'width': ref_w, 'height': ref_h})
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(data)
    return out_path

aligned_files = {}
for name, fname in files.items():
    src  = os.path.join(DATA_DIR, fname)
    out  = os.path.join(ALIGNED_DIR, f'aligned_{fname}')
    aligned_files[name] = align_raster(src, ref_path, out)
    print(f'  aligned: {name}')
print('Done')

In [ ]:
# ============================================================
# LOAD RASTERS
# ============================================================
def read_raster(path):
    with rasterio.open(path) as src:
        data   = src.read(1).astype('float32')
        nodata = src.nodata
        profile = src.profile
    if nodata is not None:
        data[data == nodata] = np.nan
    data[data <= -9999] = np.nan
    return data, profile

rasters, ref_profile = {}, None
for name, path in aligned_files.items():
    rasters[name], p = read_raster(path)
    if ref_profile is None: ref_profile = p
    print(f'{name:<20} min={np.nanmin(rasters[name]):8.2f}  max={np.nanmax(rasters[name]):8.2f}  shape={rasters[name].shape}')

rows, cols = rasters['elevation'].shape
print(f'\nGrid: {rows} x {cols}')

In [ ]:
# ============================================================
# NORMALISE 0-1
# invert=True where LOW value = HIGH flood risk
# ============================================================
def norm(arr, invert=False):
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    out = (arr - mn) / (mx - mn + 1e-10)
    return (1 - out) if invert else out

normalised = {
    'rainfall':        norm(rasters['rainfall'],        invert=False),
    'elevation':       norm(rasters['elevation'],       invert=True),
    'dist_river':      norm(rasters['dist_river'],      invert=True),
    'slope':           norm(rasters['slope'],           invert=True),
    'twi':             norm(rasters['twi'],             invert=False),
    'ndvi':            norm(rasters['ndvi'],            invert=True),
    'dist_impervious': norm(rasters['dist_impervious'], invert=True),
    'clay':            norm(rasters['clay'],            invert=False),
}

nodata_mask = np.zeros((rows, cols), dtype=bool)
for arr in normalised.values():
    nodata_mask |= np.isnan(arr)

print(f'Valid pixels : {(~nodata_mask).sum():,}')
print(f'NoData pixels: {nodata_mask.sum():,}')

In [ ]:
# ============================================================
# LOAD TRAINING CSV
# ============================================================
df = pd.read_csv(TRAINING_CSV)
df = df.drop(columns=[c for c in df.columns if c.startswith('.geo') or c == 'system:index'])
df = df.rename(columns={'rain_p95': 'rainfall', 'dist_to_river': 'dist_river'})

feature_cols = [c for c in ['elevation','slope','twi','rainfall','ndvi',
                              'dist_river','dist_impervious','clay'] if c in df.columns]
print(f'Shape: {df.shape}  |  Features: {feature_cols}')
print(f'Class balance:\n{df["label"].value_counts()}')

df = df[feature_cols + ['label']].dropna()

In [ ]:
# ============================================================
# BALANCE CLASSES & TRAIN/TEST SPLIT
# ============================================================
df_maj = df[df['label'] == 0]
df_min = df[df['label'] == 1]
df_min_up = resample(df_min, replace=True, n_samples=len(df_maj), random_state=42)
df_bal = pd.concat([df_maj, df_min_up]).sample(frac=1, random_state=42).reset_index(drop=True)

X = df_bal[feature_cols].values
y = df_bal['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]}  Test: {X_test.shape[0]}')

In [ ]:
# ============================================================
# RANDOM FOREST
# ============================================================
rf = RandomForestClassifier(
    n_estimators=200, max_features='sqrt', min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs  = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')
print(f'5-fold CV AUC: {aucs.mean():.4f} +/- {aucs.std():.4f}')

In [ ]:
# ============================================================
# VALIDATION PLOTS
# ============================================================
y_pred      = rf.predict(X_test)
y_prob      = rf.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)

print(f'Test ROC-AUC: {auc:.4f}')
print(classification_report(y_test, y_pred, target_names=['Non-flood','Flood']))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1); axes[0].set_title('ROC Curve'); axes[0].legend()
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
    display_labels=['Non-flood','Flood']).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix')
axes[2].bar(range(1,6), aucs, color='steelblue', alpha=0.7)
axes[2].axhline(aucs.mean(), color='red', linestyle='--', label=f'Mean={aucs.mean():.3f}')
axes[2].set_title('5-Fold CV AUC'); axes[2].legend(); axes[2].set_ylim(0,1)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'validation_plots.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# SHAP FEATURE IMPORTANCE
# ============================================================
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
shap_flood  = shap_values[1] if isinstance(shap_values, list) else shap_values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.sca(axes[0]); shap.summary_plot(shap_flood, X_test, feature_names=feature_cols, plot_type='bar', show=False)
axes[0].set_title('SHAP Feature Importance')
plt.sca(axes[1]); shap.summary_plot(shap_flood, X_test, feature_names=feature_cols, show=False)
axes[1].set_title('SHAP Beeswarm')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'shap_importance.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# PREDICT OVER FULL RASTER (chunked)
# ============================================================
layer_order = feature_cols
X_raster    = np.stack([normalised[c].flatten() for c in layer_order], axis=1)
nan_mask    = np.any(np.isnan(X_raster), axis=1)
valid_idx   = np.where(~nan_mask)[0]
prob_flat   = np.full(rows * cols, np.nan, dtype='float32')

for i in range(0, len(valid_idx), 100_000):
    idx = valid_idx[i:i+100_000]
    prob_flat[idx] = rf.predict_proba(X_raster[idx])[:, 1]
    print(f'  {min(i+100000, len(valid_idx)):,} / {len(valid_idx):,}', end='\r')

prob_map = prob_flat.reshape(rows, cols)

def save_tif(array, path, dtype='float32', nodata=-9999):
    meta = ref_profile.copy()
    meta.update({'count':1,'dtype':dtype,'nodata':nodata})
    out = array.copy(); out[np.isnan(out)] = nodata
    with rasterio.open(path, 'w', **meta) as dst:
        dst.write(out.astype(dtype), 1)
    print(f'  Saved: {os.path.basename(path)}')

save_tif(prob_map, os.path.join(DATA_DIR, 'RF_flood_probability_nairobi.tif'))

# Save normalised layers and nodata mask for notebook 2
import pickle
with open(os.path.join(DATA_DIR, 'model_state.pkl'), 'wb') as f:
    pickle.dump({'normalised': normalised, 'nodata_mask': nodata_mask,
                 'ref_profile': ref_profile, 'rows': rows, 'cols': cols,
                 'rasters': rasters}, f)
print('\nmodel_state.pkl saved for notebook 2')